# Criação do índice de sentimentos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score
from nltk.corpus import stopwords
from nltk import ngrams
from nltk.tokenize import word_tokenize
from collections import Counter
import spacy
nlp = spacy.load('pt_core_news_sm')

In [2]:
# Carregando dados
df = pd.read_csv('dados/folhamercado_tweets.csv', usecols=['date', 'user', 'tweet'])

df.head()

,date,user,tweet
0,2011-01-01,folha_mercado,Lojas de shoppings centers estão proibidos de ...
1,2011-01-01,folha_mercado,"VIDEO: Governo vai ter que ""se esforçar"" para ..."
2,2011-01-01,folha_mercado,Veja as manchetes dos principais jornais deste...
3,2011-01-01,folha_mercado,"De férias, Obama diz que sua meta para 2011 é ..."
4,2011-01-01,folha_mercado,Governo prorroga até 2035 encargo na conta de ...


In [11]:
# Pré-processamento
import re
import string

df['date'] = pd.to_datetime(df['date'])

df['tweet'] = df['tweet'].apply(lambda x: re.sub("http\S+", "", x))
df['tweet'] = df['tweet'].apply(lambda x: re.sub(r'#\w+', '', x))

stop_words = set(stopwords.words('portuguese'))
stop_words.update(['em', 'um', 'uma', 'a', 'as', 'o', 'os', 'de', 'do', 'da', 'dos', 'das', 'e', 'que', 'com', 
                   'para', 'por', 'no', 'na', 'nos', 'nas', 'em', 'este', 'esta', 'esse', 'essa', 'neste', 'nesta'])

df['tweet_clean'] = df['tweet'].str.lower() 
df['tweet_clean'] = df['tweet_clean'].str.translate(str.maketrans('', '', string.punctuation))
df['tweet_clean'] = df['tweet_clean'].str.replace(' r ', '')
df['tweet_clean'] = df['tweet_clean'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))

df.head()

<>:7: SyntaxWarning: invalid escape sequence '\S'
<>:7: SyntaxWarning: invalid escape sequence '\S'
C:\Users\luans\AppData\Local\Temp\ipykernel_7936\1880929333.py:7: SyntaxWarning: invalid escape sequence '\S'
  df['tweet'] = df['tweet'].apply(lambda x: re.sub("http\S+", "", x))


,date,user,tweet,tweet_clean,tweet_lemma
0,2011-01-01,folha_mercado,Lojas de shoppings centers estão proibidos de ...,lojas shoppings centers proibidos abrir hoje,loja Shoppings centers proibir abrir hoje
1,2011-01-01,folha_mercado,"VIDEO: Governo vai ter que ""se esforçar"" para ...",video governo vai ter esforçar combater inflaç...,video governo ir ter esforçar combater inflaçã...
2,2011-01-01,folha_mercado,Veja as manchetes dos principais jornais deste...,veja manchetes principais jornais deste sábado,ver manchete principal jornal de este sábado
3,2011-01-01,folha_mercado,"De férias, Obama diz que sua meta para 2011 é ...",férias obama diz meta 2011 recuperar economia,férias obama dizer meta 2011 recuperar economia
4,2011-01-01,folha_mercado,Governo prorroga até 2035 encargo na conta de ...,governo prorroga 2035 encargo conta luz,governo prorrogar 2035 encargo contar luz


In [12]:
# Lemmatization
lemma_docs = nlp.pipe(df['tweet_clean'])

df['tweet_lemma'] = [' '.join([token.lemma_ for token in doc]) for doc in lemma_docs]

In [13]:
df.to_csv('dados/fm_tweets_preprocessed.csv', index=False)

## Análise de Dados

In [5]:
# extraindo n-grams
from functions import get_ngrams
from collections import Counter

df_range = df[(df['date'] >= '2015-01-01') & (df['date'] <= '2019-12-31')]

bigrams = get_ngrams(df_range['tweet_clean'], 2)
trigrams = get_ngrams(df_range['tweet_clean'], 3)

freq_bigram = Counter(bigrams)
freq_trigram = Counter(trigrams)

print('Bi-grans mais comuns:')
for n, freq in enumerate(freq_bigram.most_common(5)):
    print(f'Bi-gram {n+1}: {freq[0]} - Frequência: {freq[1]}')

print('=' * 20)

print('Tri-grans mais comuns:')
for n, freq in enumerate(freq_trigram.most_common(5)):
    print(f'Tri-gram {n+1}: {freq[0]} - Frequência: {freq[1]}')

Bi-grans mais comuns:
Bi-gram 1: ('reforma', 'previdência') - Frequência: 605
Bi-gram 2: ('diz', 'presidente') - Frequência: 326
Bi-gram 3: ('dólar', 'cai') - Frequência: 269
Bi-gram 4: ('banco', 'central') - Frequência: 266
Bi-gram 5: ('bolsa', 'sobe') - Frequência: 255
Tri-grans mais comuns:
Tri-gram 1: ('sobre', 'reforma', 'previdência') - Frequência: 34
Tri-gram 2: ('reforma', 'previdência', 'diz') - Frequência: 33
Tri-gram 3: ('contra', 'reforma', 'previdência') - Frequência: 32
Tri-gram 4: ('taxa', 'básica', 'juros') - Frequência: 31
Tri-gram 5: ('pedido', 'recuperação', 'judicial') - Frequência: 28


In [14]:
df_range = df[(df['date'] >= '2015-01-01') & (df['date'] <= '2019-12-31')]

bigrams = get_ngrams(df_range['tweet_lemma'], 2)
trigrams = get_ngrams(df_range['tweet_lemma'], 3)

freq_bigram = Counter(bigrams)
freq_trigram = Counter(trigrams)

print('Bi-grans mais comuns:')
for n, freq in enumerate(freq_bigram.most_common(5)):
    print(f'Bi-gram {n+1}: {freq[0]} - Frequência: {freq[1]}')

print('=' * 20)

print('Tri-grans mais comuns:')
for n, freq in enumerate(freq_trigram.most_common(5)):
    print(f'Tri-gram {n+1}: {freq[0]} - Frequência: {freq[1]}')

Bi-grans mais comuns:
Bi-gram 1: ('reforma', 'previdência') - Frequência: 444
Bi-gram 2: ('dizer', 'presidente') - Frequência: 327
Bi-gram 3: ('banco', 'central') - Frequência: 280
Bi-gram 4: ('dólar', 'cair') - Frequência: 274
Bi-gram 5: ('dólar', 'subir') - Frequência: 264
Tri-grans mais comuns:
Tri-gram 1: ('pedir', 'recuperação', 'judicial') - Frequência: 50
Tri-gram 2: ('sobre', 'reforma', 'previdência') - Frequência: 34
Tri-gram 3: ('contra', 'reforma', 'previdência') - Frequência: 32
Tri-gram 4: ('taxa', 'básico', 'juro') - Frequência: 31
Tri-gram 5: ('de', 'este', 'ano') - Frequência: 31


## Aplicação do modelo

In [ ]:
# Carregando modelo

import joblib
model = joblib.load('modelos/naive_bayes_sentiment_analysis_ptbr.pkl')

x = df_range['tweet_lemma']
y_pred = model.predict(x)